# Project 2B: Dead Reckoning on RealReplay

This notebook runs IMU-only dead reckoning on a selected bag in `data/` (start with `building.bag`).


In [1]:
import numpy as np
import gtsam
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from measurement import Measurement
from real_replay import RealReplay


In [5]:
bag_path = "data/building.bag"
r = RealReplay(bag_path)

preint_params = gtsam.PreintegrationParams.MakeSharedU(9.81)
# Keep these simple and explicit for students.
sigma_gyro = 8e-4
sigma_acc = 1e-2
sigma_integration = 1e-3
preint_params.setGyroscopeCovariance(np.diag([sigma_gyro**2] * 3))
preint_params.setAccelerometerCovariance(np.diag([sigma_acc**2] * 3))
preint_params.setIntegrationCovariance(np.diag([sigma_integration] * 3))

x0 = gtsam.NavState(gtsam.Rot3(), np.zeros(3), np.zeros(3))
rot_var_init = np.array([1e-2, 1e-2, 1e-6], dtype=float)
P0 = np.diag(np.concatenate([rot_var_init, [0.05] * 3, [0.05] * 3]))

filt = gtsam.NavStateImuEKF(x0, P0, preint_params)

p_hist = []
ypr_hist = []
forward_vel_hist = []


def dr_callback(m: Measurement) -> None:
    if m.dt > 0.0:
        filt.predict(m.omega_meas, m.f_meas, m.dt)
    x = filt.state()
    p_hist.append(np.asarray(x.position(), dtype=float))
    ypr_hist.append(np.asarray(x.attitude().ypr(), dtype=float))
    forward_vel_hist.append(float((x.attitude().matrix().T @ np.asarray(x.velocity(), dtype=float))[0]))


r.replay(dr_callback)
r.close()

p_hist = np.asarray(p_hist)
ypr_hist = np.asarray(ypr_hist)
forward_vel_hist = np.asarray(forward_vel_hist)

print(f"Dead-reckoning samples: {len(p_hist)}")


Dead-reckoning samples: 30354


In [6]:
k = np.arange(len(p_hist))

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("Position (x,y,z)", "Yaw-Pitch-Roll", "Forward velocity"))

fig.add_trace(go.Scatter(x=k, y=p_hist[:, 0], name="x"), row=1, col=1)
fig.add_trace(go.Scatter(x=k, y=p_hist[:, 1], name="y"), row=1, col=1)
fig.add_trace(go.Scatter(x=k, y=p_hist[:, 2], name="z"), row=1, col=1)

fig.add_trace(go.Scatter(x=k, y=ypr_hist[:, 0], name="yaw"), row=2, col=1)
fig.add_trace(go.Scatter(x=k, y=ypr_hist[:, 1], name="pitch"), row=2, col=1)
fig.add_trace(go.Scatter(x=k, y=ypr_hist[:, 2], name="roll"), row=2, col=1)

fig.add_trace(go.Scatter(x=k, y=forward_vel_hist, name="forward vel"), row=3, col=1)

fig.update_layout(height=800, width=1000, title="Dead Reckoning on {bag_path}")
fig.update_xaxes(title_text="sample index", row=3, col=1)
fig.show()
